In [1]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import requests
import xgboost as xgb

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
)

PROJECT_ROOT = Path.cwd().resolve().parents[1]
FLIGHTS_PATH = PROJECT_ROOT / "src/data/processed/flights_2024.parquet"
WEATHER_DIR = PROJECT_ROOT / "src/data/external/noaa_isd"
OUTPUT_DIR = PROJECT_ROOT / "artifacts/weather_v2"

WEATHER_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert FLIGHTS_PATH.exists(), f"Missing file: {FLIGHTS_PATH}"

airport_station_ids = {
    "ATL": "72219013874",
    "DFW": "72259003927",
    "DEN": "72565003017",
    "ORD": "72530094846",
    "CLT": "72314013881",
    "LAX": "72295023174",
    "LAS": "72386023169",
    "SEA": "72793024233",
    "PHX": "72278023183",
    "LGA": "72503014732",
}

airport_timezones = {
    "ATL": "America/New_York",
    "DFW": "America/Chicago",
    "DEN": "America/Denver",
    "ORD": "America/Chicago",
    "CLT": "America/New_York",
    "LAX": "America/Los_Angeles",
    "LAS": "America/Los_Angeles",
    "SEA": "America/Los_Angeles",
    "PHX": "America/Phoenix",
    "LGA": "America/New_York",
}

flights = pd.read_parquet(FLIGHTS_PATH)
flights["FlightDate"] = pd.to_datetime(
    flights["FlightDate"], errors="coerce"
)

required = {"FlightDate", "Origin", "CRSDepTime", "DepDel15"}
missing = required - set(flights.columns)
assert not missing, f"Missing required columns: {missing}"

flights = flights[
    flights["Origin"].isin(airport_station_ids)
    & flights["FlightDate"].between("2024-01-01", "2024-12-31")
].copy()

flights["DepDel15"] = pd.to_numeric(
    flights["DepDel15"], errors="coerce"
)
flights = flights[flights["DepDel15"].isin([0, 1])].copy()
flights["DepDel15"] = flights["DepDel15"].astype("int8")
flights = flights.reset_index(drop=True)
flights["_flight_id"] = np.arange(len(flights))

print(f"Flight rows: {len(flights):,}")
print("Airports:", sorted(flights["Origin"].unique()))

Flight rows: 2,336,387
Airports: ['ATL', 'CLT', 'DEN', 'DFW', 'LAS', 'LAX', 'LGA', 'ORD', 'PHX', 'SEA']


In [2]:
hhmm = pd.to_numeric(flights["CRSDepTime"], errors="coerce")
hour = np.floor(hhmm / 100)
minute = hhmm % 100

valid_hhmm = (
    hhmm.notna()
    & hhmm.between(0, 2400)
    & minute.lt(60)
    & ((hhmm.ne(2400)) | minute.eq(0))
)

hour = hour.where(hhmm.ne(2400), 0)
minute = minute.where(hhmm.ne(2400), 0)

local_departure = (
    flights["FlightDate"].dt.normalize()
    + pd.to_timedelta(hour, unit="h")
    + pd.to_timedelta(minute, unit="m")
).where(valid_hhmm)

flights["scheduled_departure_utc"] = pd.Series(
    pd.NaT, index=flights.index, dtype="datetime64[ns, UTC]"
)

for airport, timezone in airport_timezones.items():
    mask = flights["Origin"].eq(airport) & local_departure.notna()

    local_aware = local_departure.loc[mask].dt.tz_localize(
        timezone, ambiguous="NaT", nonexistent="NaT"
    )

    flights.loc[mask, "scheduled_departure_utc"] = (
        local_aware.dt.tz_convert("UTC")
    )

flights["weather_cutoff_utc"] = (
    flights["scheduled_departure_utc"] - pd.Timedelta(hours=2)
)

print(
    "Missing scheduled departure UTC:",
    f"{flights['scheduled_departure_utc'].isna().sum():,}",
)

Missing scheduled departure UTC: 4


In [7]:

weather_parts = []
download_log = []

for airport, station_id in airport_station_ids.items():
    cache_path = WEATHER_DIR / f"{station_id}_2024.csv"
    url = (
        "https://noaa-global-hourly-pds.s3.amazonaws.com/"
        f"2024/{station_id}.csv"
    )

    # Download if missing or if the existing cache is invalid.
    def read_noaa_csv(path):
        # NOAA CSVs can contain malformed rows; skip those rows.
        df = pd.read_csv(
            path,
            dtype=str,
            engine="python",
            on_bad_lines="skip",
        )

        # A valid NOAA station file must contain DATE.
        if "DATE" not in df.columns:
            raise ValueError(
                f"Invalid NOAA CSV: DATE column missing in {path.name}. "
                f"Columns found: {df.columns.tolist()[:10]}"
            )
        return df

    obs = None

    if cache_path.exists():
        try:
            obs = read_noaa_csv(cache_path)
        except Exception as e:
            print(f"Invalid cached file {cache_path.name}: {e}")
            cache_path.unlink(missing_ok=True)

    if obs is None:
        response = requests.get(url, timeout=120)
        response.raise_for_status()

        # Avoid caching HTML/error responses as CSV files.
        if (
            "<html" in response.text[:1000].lower()
            or "<!doctype html" in response.text[:1000].lower()
        ):
            raise ValueError(
                f"NOAA returned HTML for {airport} ({station_id}). "
                "Check the URL or network response."
            )

        cache_path.write_text(response.text, encoding="utf-8")

        try:
            obs = read_noaa_csv(cache_path)
        except Exception:
            cache_path.unlink(missing_ok=True)
            raise

    obs["DATE"] = pd.to_datetime(
        obs["DATE"], errors="coerce", utc=True
    )

    obs = obs[
        obs["DATE"].ge(pd.Timestamp("2024-01-01", tz="UTC"))
        & obs["DATE"].lt(pd.Timestamp("2025-01-01", tz="UTC"))
    ].copy()

    obs["station_id"] = station_id
    obs["airport"] = airport

    weather_parts.append(obs)

    download_log.append({
        "airport": airport,
        "station_id": station_id,
        "rows_2024": len(obs),
        "cache": str(cache_path),
    })

    print(f"{airport}: {len(obs):,} observations loaded")

weather_raw = pd.concat(weather_parts, ignore_index=True)

display(pd.DataFrame(download_log))
print(f"Total NOAA observations: {len(weather_raw):,}")


ATL: 13,448 observations loaded
Invalid cached file 72259003927_2024.csv: Invalid NOAA CSV: DATE column missing in 72259003927_2024.csv. Columns found: ['<!DOCTYPE html>']
DFW: 13,458 observations loaded
Invalid cached file 72565003017_2024.csv: Invalid NOAA CSV: DATE column missing in 72565003017_2024.csv. Columns found: ['<!DOCTYPE html>']
DEN: 13,421 observations loaded
Invalid cached file 72530094846_2024.csv: Invalid NOAA CSV: DATE column missing in 72530094846_2024.csv. Columns found: ['<!DOCTYPE html>']
ORD: 14,041 observations loaded
Invalid cached file 72314013881_2024.csv: Invalid NOAA CSV: DATE column missing in 72314013881_2024.csv. Columns found: ['<!DOCTYPE html>']
CLT: 13,820 observations loaded
Invalid cached file 72295023174_2024.csv: Invalid NOAA CSV: DATE column missing in 72295023174_2024.csv. Columns found: ['<!DOCTYPE html>']
LAX: 14,049 observations loaded
Invalid cached file 72386023169_2024.csv: Invalid NOAA CSV: DATE column missing in 72386023169_2024.csv. Col

,airport,station_id,rows_2024,cache
0,ATL,72219013874,13448,/Users/apple/Desktop/Code/ai_engnr_learning/pr...
1,DFW,72259003927,13458,/Users/apple/Desktop/Code/ai_engnr_learning/pr...
2,DEN,72565003017,13421,/Users/apple/Desktop/Code/ai_engnr_learning/pr...
3,ORD,72530094846,14041,/Users/apple/Desktop/Code/ai_engnr_learning/pr...
4,CLT,72314013881,13820,/Users/apple/Desktop/Code/ai_engnr_learning/pr...
5,LAX,72295023174,14049,/Users/apple/Desktop/Code/ai_engnr_learning/pr...
6,LAS,72386023169,12345,/Users/apple/Desktop/Code/ai_engnr_learning/pr...
7,SEA,72793024233,14192,/Users/apple/Desktop/Code/ai_engnr_learning/pr...
8,PHX,72278023183,12246,/Users/apple/Desktop/Code/ai_engnr_learning/pr...
9,LGA,72503014732,13414,/Users/apple/Desktop/Code/ai_engnr_learning/pr...


Total NOAA observations: 134,434


In [10]:
# Inspect variables created by the NOAA loading cell

for name, value in list(globals().items()):
    if isinstance(value, pd.DataFrame):
        print(f"{name}: shape={value.shape}, columns={value.columns.tolist()[:12]}")

flights: shape=(2336387, 21), columns=['Month', 'FlightDate', 'Reporting_Airline', 'Origin', 'Dest', 'CRSDepTime', 'DepDel15', 'CRSElapsedTime', 'Distance', 'DepHour', 'DepMinute', 'FlightDay']
obs: shape=(13414, 103), columns=['STATION', 'DATE', 'SOURCE', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'NAME', 'REPORT_TYPE', 'CALL_SIGN', 'QUALITY_CONTROL', 'WND', 'CIG']
weather_raw: shape=(134434, 108), columns=['STATION', 'DATE', 'SOURCE', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'NAME', 'REPORT_TYPE', 'CALL_SIGN', 'QUALITY_CONTROL', 'WND', 'CIG']


In [11]:
# Cell 4 — Validate NOAA observations and station coverage

import pandas as pd

print("Weather data shape:", weather_raw.shape)
print("Unique stations:", weather_raw["STATION"].nunique())
print("Missing timestamps:", weather_raw["DATE"].isna().sum())

# Convert NOAA observation timestamps to UTC
weather_raw["DATE"] = pd.to_datetime(
    weather_raw["DATE"],
    errors="coerce",
    utc=True,
)

# Summary of observation coverage by station
station_summary = (
    weather_raw.groupby(["STATION", "NAME"], dropna=False)
    .agg(
        observations=("DATE", "count"),
        first_observation=("DATE", "min"),
        last_observation=("DATE", "max"),
    )
    .reset_index()
    .sort_values("observations", ascending=False)
)

display(station_summary)

Weather data shape: (134434, 108)
Unique stations: 10
Missing timestamps: 0


,STATION,NAME,observations,first_observation,last_observation
9,72793024233,"SEATTLE TACOMA AIRPORT, WA US",14192,2024-01-01 00:00:00+00:00,2024-12-31 23:53:00+00:00
3,72295023174,"LOS ANGELES INTERNATIONAL AIRPORT, CA US",14049,2024-01-01 00:00:00+00:00,2024-12-31 23:53:00+00:00
7,72530094846,"CHICAGO OHARE INTERNATIONAL AIRPORT, IL US",14041,2024-01-01 00:00:00+00:00,2024-12-31 23:51:00+00:00
4,72314013881,"CHARLOTTE DOUGLAS AIRPORT, NC US",13820,2024-01-01 00:00:00+00:00,2024-12-31 23:52:00+00:00
1,72259003927,"DAL FTW WSCMO AIRPORT, TX US",13458,2024-01-01 00:00:00+00:00,2024-12-31 23:53:00+00:00
0,72219013874,ATLANTA HARTSFIELD JACKSON INTERNATIONAL AIRPO...,13448,2024-01-01 00:00:00+00:00,2024-12-31 23:52:00+00:00
8,72565003017,"DENVER INTERNATIONAL AIRPORT, CO US",13421,2024-01-01 00:00:00+00:00,2024-12-31 23:53:00+00:00
6,72503014732,"LAGUARDIA AIRPORT, NY US",13414,2024-01-01 00:00:00+00:00,2024-12-31 23:51:00+00:00
5,72386023169,"MCCARRAN INTERNATIONAL AIRPORT, NV US",12345,2024-01-01 00:00:00+00:00,2024-12-31 23:56:00+00:00
2,72278023183,"PHOENIX AIRPORT, AZ US",12246,2024-01-01 00:00:00+00:00,2024-12-31 23:51:00+00:00


In [12]:
# Cell 5 — Parse NOAA Global Hourly measurements

import numpy as np

def parse_noaa_measurement(value, field):
    if pd.isna(value):
        return np.nan

    parts = str(value).split(",")

    try:
        if field in {"TMP", "DEW"}:
            raw = parts[0].strip()
            quality = parts[1].strip() if len(parts) > 1 else ""
            if len(raw) < 5 or quality not in {"1", "5"}:
                return np.nan
            number = int(raw)
            if number in {-9999, 9999}:
                return np.nan
            return number / 10

        if field == "SLP":
            raw = parts[0].strip()
            quality = parts[1].strip() if len(parts) > 1 else ""
            if quality not in {"1", "5"}:
                return np.nan
            number = int(raw)
            if number == 99999:
                return np.nan
            return number / 10

        if field == "WND":
            if len(parts) < 5:
                return np.nan
            speed = int(parts[3])
            quality = parts[4].strip()
            if quality not in {"1", "5"} or speed == 9999:
                return np.nan
            return speed / 10

        if field == "VIS":
            if len(parts) < 3:
                return np.nan
            distance = int(parts[0])
            quality = parts[2].strip()
            if quality not in {"1", "5"} or distance == 999999:
                return np.nan
            return distance / 1000

        if field == "CIG":
            if len(parts) < 2:
                return np.nan
            height = int(parts[0])
            quality = parts[1].strip()
            if quality not in {"1", "5"} or height == 99999:
                return np.nan
            return height * 0.3048  # feet to metres

    except (ValueError, IndexError):
        return np.nan

    return np.nan


weather_fields = {
    "TMP": "temp_c",
    "DEW": "dewpoint_c",
    "SLP": "pressure_hpa",
    "WND": "wind_speed_ms",
    "VIS": "visibility_km",
    "CIG": "ceiling_m",
}

for raw_col, output_col in weather_fields.items():
    if raw_col in weather_raw.columns:
        weather_raw[output_col] = weather_raw[raw_col].map(
            lambda value, field=raw_col:
                parse_noaa_measurement(value, field)
        )
    else:
        weather_raw[output_col] = np.nan

weather_columns = list(weather_fields.values())

display(weather_raw[["STATION", "DATE"] + weather_columns].head())
display(weather_raw[weather_columns].describe().T)

,STATION,DATE,temp_c,dewpoint_c,pressure_hpa,wind_speed_ms,visibility_km,ceiling_m
0,72219013874,2024-01-01 00:00:00+00:00,9.4,0.0,1020.4,2.1,NaN,6705.6000
1,72219013874,2024-01-01 00:52:00+00:00,8.9,0.0,1020.3,2.6,NaN,6705.6000
2,72219013874,2024-01-01 01:52:00+00:00,7.8,0.0,1020.4,2.1,NaN,6705.6000
3,72219013874,2024-01-01 02:52:00+00:00,7.8,0.0,1020.2,2.6,NaN,2043.9888
4,72219013874,2024-01-01 03:00:00+00:00,7.8,0.0,1020.2,2.6,NaN,NaN


,count,mean,std,min,25%,50%,75%,max
temp_c,130524.0,16.871241,10.192327,-26.7,10.0000,16.7,23.3,48.3
dewpoint_c,130536.0,6.752919,9.305906,-31.7,0.0000,7.2,14.4,26.1
pressure_hpa,115157.0,1014.983084,6.944946,981.0,1010.5000,1015.0,1019.3,1048.6
wind_speed_ms,130462.0,3.761905,2.305174,0.0,2.1000,3.6,5.1,25.2
visibility_km,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ceiling_m,113100.0,3899.068931,2995.467660,0.0,464.5152,6705.6,6705.6,6705.6


In [13]:
# Cell 6 — Missingness and range checks

print("Missing percentage by weather feature:")
display((weather_raw[weather_columns].isna().mean() * 100).round(2))

print("\nBasic range checks:")
for col in weather_columns:
    values = weather_raw[col].dropna()
    if not values.empty:
        print(
            f"{col}: min={values.min():.2f}, "
            f"median={values.median():.2f}, "
            f"max={values.max():.2f}"
        )

Missing percentage by weather feature:


temp_c             2.91
dewpoint_c         2.90
pressure_hpa      14.34
wind_speed_ms      2.95
visibility_km    100.00
ceiling_m         15.87
dtype: float64


Basic range checks:
temp_c: min=-26.70, median=16.70, max=48.30
dewpoint_c: min=-31.70, median=7.20, max=26.10
pressure_hpa: min=981.00, median=1015.00, max=1048.60
wind_speed_ms: min=0.00, median=3.60, max=25.20
ceiling_m: min=0.00, median=6705.60, max=6705.60


In [14]:
# Cell 7 — Attach NOAA station IDs to flights

# Reuse the station mapping from your earlier cell.
# It must map airport codes to NOAA station IDs.
print("Station mapping variable candidates:")

for name, value in list(globals().items()):
    if isinstance(value, dict) and any(
        airport in value for airport in ["ATL", "DFW", "DEN", "ORD"]
    ):
        print(name, value)

Station mapping variable candidates:
airport_station_ids {'ATL': '72219013874', 'DFW': '72259003927', 'DEN': '72565003017', 'ORD': '72530094846', 'CLT': '72314013881', 'LAX': '72295023174', 'LAS': '72386023169', 'SEA': '72793024233', 'PHX': '72278023183', 'LGA': '72503014732'}
airport_timezones {'ATL': 'America/New_York', 'DFW': 'America/Chicago', 'DEN': 'America/Denver', 'ORD': 'America/Chicago', 'CLT': 'America/New_York', 'LAX': 'America/Los_Angeles', 'LAS': 'America/Los_Angeles', 'SEA': 'America/Los_Angeles', 'PHX': 'America/Phoenix', 'LGA': 'America/New_York'}


In [15]:
# Cell 8 — Prepare data for a leakage-safe weather join

from pathlib import Path
import pandas as pd
import numpy as np

required_flight_cols = {"Origin", "weather_cutoff_utc"}
missing = required_flight_cols - set(flights.columns)

if missing:
    raise ValueError(f"Missing required flight columns: {sorted(missing)}")

# Assign the NOAA station to each flight's origin airport.
flights_weather = flights.copy()
flights_weather["_flight_row_id"] = np.arange(len(flights_weather))
flights_weather["STATION"] = flights_weather["Origin"].map(
    airport_station_ids
).astype("string")

flights_weather["weather_cutoff_utc"] = pd.to_datetime(
    flights_weather["weather_cutoff_utc"],
    errors="coerce",
    utc=True,
)

# Prepare weather observations.
weather_join = weather_raw.copy()
weather_join["STATION"] = weather_join["STATION"].astype("string")
weather_join["DATE"] = pd.to_datetime(
    weather_join["DATE"], errors="coerce", utc=True
)

weather_features = [
    "temp_c",
    "dewpoint_c",
    "pressure_hpa",
    "wind_speed_ms",
    "visibility_km",
    "ceiling_m",
]

# Retain only stations in our mapping and valid timestamps.
valid_stations = set(airport_station_ids.values())

weather_join = weather_join.loc[
    weather_join["STATION"].isin(valid_stations)
    & weather_join["DATE"].notna()
].copy()

# Remove duplicate station/timestamp records to make the join deterministic.
weather_join = (
    weather_join.sort_values("DATE")
    .drop_duplicates(["STATION", "DATE"], keep="last")
)

print("Flights:", len(flights_weather))
print("Valid weather observations:", len(weather_join))
print("Flights without a mapped station:", flights_weather["STATION"].isna().sum())
print("Flights without a valid cutoff:", flights_weather["weather_cutoff_utc"].isna().sum())
print("Weather stations:", weather_join["STATION"].nunique())

display(
    flights_weather[["Origin", "STATION", "weather_cutoff_utc"]].head()
)

Flights: 2336387
Valid weather observations: 134196
Flights without a mapped station: 0
Flights without a valid cutoff: 4
Weather stations: 10


,Origin,STATION,weather_cutoff_utc
0,LGA,72503014732,2024-01-08 11:56:00+00:00
1,LGA,72503014732,2024-01-09 11:56:00+00:00
2,LGA,72503014732,2024-01-10 11:56:00+00:00
3,LGA,72503014732,2024-01-11 11:56:00+00:00
4,LGA,72503014732,2024-01-12 11:56:00+00:00


In [18]:

# Cell 9 — Leakage-safe backward as-of join (fixed datetime precision)

# Normalize both merge keys to the same dtype.
flights_valid = flights_weather.loc[
    flights_weather["STATION"].notna()
    & flights_weather["weather_cutoff_utc"].notna()
].copy()

flights_valid["weather_cutoff_utc"] = pd.to_datetime(
    flights_valid["weather_cutoff_utc"], utc=True
).astype("datetime64[ns, UTC]")

weather_for_join = weather_join[
    ["STATION", "DATE"] + weather_features
].copy()

weather_for_join = weather_for_join.rename(
    columns={"DATE": "weather_obs_time_utc"}
)

weather_for_join["weather_obs_time_utc"] = pd.to_datetime(
    weather_for_join["weather_obs_time_utc"], utc=True
).astype("datetime64[ns, UTC]")

# Ensure station IDs use the same dtype on both sides.
flights_valid["STATION"] = flights_valid["STATION"].astype("string")
weather_for_join["STATION"] = weather_for_join["STATION"].astype("string")

# merge_asof requires the time key to be globally sorted.
left = flights_valid.sort_values("weather_cutoff_utc").reset_index(drop=True)
right = weather_for_join.sort_values("weather_obs_time_utc").reset_index(drop=True)

print("Flight cutoff dtype:", left["weather_cutoff_utc"].dtype)
print("Weather timestamp dtype:", right["weather_obs_time_utc"].dtype)

flights_with_weather = pd.merge_asof(
    left,
    right,
    left_on="weather_cutoff_utc",
    right_on="weather_obs_time_utc",
    by="STATION",
    direction="backward",
    tolerance=pd.Timedelta("2 hours"),
)

flights_with_weather = (
    flights_with_weather
    .sort_values("_flight_row_id")
    .reset_index(drop=True)
)

# Leakage checks
matched = flights_with_weather["weather_obs_time_utc"].notna()

future_weather = (
    flights_with_weather.loc[matched, "weather_obs_time_utc"]
    > flights_with_weather.loc[matched, "weather_cutoff_utc"]
).sum()

assert future_weather == 0, "Leakage detected: future weather was matched!"

print("Matched flights:", int(matched.sum()))
print("Unmatched flights:", int((~matched).sum()))
print("Weather match rate:", f"{matched.mean():.2%}")
print("Future-weather matches:", future_weather)


Flight cutoff dtype: datetime64[ns, UTC]
Weather timestamp dtype: datetime64[ns, UTC]
Matched flights: 2334039
Unmatched flights: 2344
Weather match rate: 99.90%
Future-weather matches: 0


In [19]:
# Cell 10 — Weather coverage by airport

coverage = (
    flights_with_weather.assign(
        weather_matched=flights_with_weather["weather_obs_time_utc"].notna()
    )
    .groupby("Origin")
    .agg(
        flights=("_flight_row_id", "size"),
        matched=("weather_matched", "sum"),
        match_rate=("weather_matched", "mean"),
    )
    .reset_index()
)

coverage["match_rate"] = (coverage["match_rate"] * 100).round(2)

display(coverage.sort_values("match_rate"))

print("\nMissing weather-feature percentage:")
display(
    (
        flights_with_weather[weather_features].isna().mean() * 100
    ).round(2).rename("missing_percent").to_frame()
)

,Origin,flights,matched,match_rate
9,SEA,161971,161528,99.73
7,ORD,276447,275956,99.82
4,LAS,187659,187343,99.83
5,LAX,192529,192260,99.86
8,PHX,192414,192175,99.88
1,CLT,214703,214573,99.94
6,LGA,158810,158720,99.94
3,DFW,307965,307826,99.95
0,ATL,338589,338459,99.96
2,DEN,305296,305199,99.97



Missing weather-feature percentage:


,missing_percent
temp_c,0.24
dewpoint_c,0.23
pressure_hpa,5.82
wind_speed_ms,0.20
visibility_km,100.00
ceiling_m,15.63


In [20]:
# Cell 11 — Save FlightDelay V2 weather features

output_dir = Path("artifacts/weather_v2")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "flights_weather_v2.parquet"
coverage_path = output_dir / "weather_coverage_by_airport.csv"

flights_with_weather.to_parquet(output_path, index=False)
coverage.to_csv(coverage_path, index=False)

print("Saved weather-enriched flights:", output_path.resolve())
print("Saved airport coverage report:", coverage_path.resolve())
print("Output shape:", flights_with_weather.shape)

Saved weather-enriched flights: /Users/apple/Desktop/Code/ai_engnr_learning/projects/flight_delay_pred/src/notebooks/artifacts/weather_v2/flights_weather_v2.parquet
Saved airport coverage report: /Users/apple/Desktop/Code/ai_engnr_learning/projects/flight_delay_pred/src/notebooks/artifacts/weather_v2/weather_coverage_by_airport.csv
Output shape: (2336383, 30)


In [21]:
# Cell 12 — Final validation

assert len(flights_with_weather) == len(flights_valid), (
    "Flight row count changed during the weather join."
)

matched = flights_with_weather["weather_obs_time_utc"].notna()

assert (
    flights_with_weather.loc[matched, "weather_obs_time_utc"]
    <= flights_with_weather.loc[matched, "weather_cutoff_utc"]
).all(), "Found weather observations after the prediction cutoff."

age_hours = (
    flights_with_weather.loc[matched, "weather_cutoff_utc"]
    - flights_with_weather.loc[matched, "weather_obs_time_utc"]
).dt.total_seconds() / 3600

assert (age_hours <= 2).all(), "Found a weather match older than two hours."

print("All leakage checks passed.")
print(f"Flights processed: {len(flights_with_weather):,}")
print(f"Weather match rate: {matched.mean():.2%}")
print(f"Weather feature columns: {weather_features}")

All leakage checks passed.
Flights processed: 2,336,383
Weather match rate: 99.90%
Weather feature columns: ['temp_c', 'dewpoint_c', 'pressure_hpa', 'wind_speed_ms', 'visibility_km', 'ceiling_m']


In [22]:
from pathlib import Path
import pandas as pd

output_dir = Path("artifacts/weather_v2")
flights_path = output_dir / "flights_weather_v2.parquet"
coverage_path = output_dir / "weather_coverage_by_airport.csv"

print("Flights artifact exists:", flights_path.exists())
print("Coverage report exists:", coverage_path.exists())

if flights_path.exists():
    saved = pd.read_parquet(flights_path)
    print("Saved shape:", saved.shape)
    print("Weather features present:", [
        c for c in [
            "temp_c", "dewpoint_c", "pressure_hpa",
            "wind_speed_ms", "visibility_km", "ceiling_m"
        ] if c in saved.columns
    ])

if coverage_path.exists():
    display(pd.read_csv(coverage_path))

Flights artifact exists: True
Coverage report exists: True
Saved shape: (2336383, 30)
Weather features present: ['temp_c', 'dewpoint_c', 'pressure_hpa', 'wind_speed_ms', 'visibility_km', 'ceiling_m']


,Origin,flights,matched,match_rate
0,ATL,338589,338459,99.96
1,CLT,214703,214573,99.94
2,DEN,305296,305199,99.97
3,DFW,307965,307826,99.95
4,LAS,187659,187343,99.83
5,LAX,192529,192260,99.86
6,LGA,158810,158720,99.94
7,ORD,276447,275956,99.82
8,PHX,192414,192175,99.88
9,SEA,161971,161528,99.73
